In [1]:
import requests
import zipfile
import pandas as pd
import io
import os
import tempfile

In [3]:
def process_backblaze_data(zip_url):
    """
    Downloads a zip file from Backblaze, extracts CSVs, and concatenates them into a single DataFrame.
    """
    print(f"Downloading {zip_url}...")
    response = requests.get(zip_url)
    response.raise_for_status() # Raise an exception for HTTP errors

    # Create a temporary directory to extract files
    with tempfile.TemporaryDirectory() as temp_dir:
        print(f"Extracting files to temporary directory: {temp_dir}")

        # Read the content into a BytesIO object and extract
        with zipfile.ZipFile(io.BytesIO(response.content)) as zf:
            zf.extractall(temp_dir)

        print("Extraction complete. Identifying and concatenating CSV files...")

        all_csv_files = []
        for root, _, files in os.walk(temp_dir):
            for file in files:
                file_path = os.path.join(root, file)
                # Exclude files from __MACOSX directory and files starting with ._
                if file_path.endswith('.csv') and '__MACOSX' not in file_path and not file.startswith('._'):
                    all_csv_files.append(file_path)

        if not all_csv_files:
            print("No CSV files found in the extracted content.")
            return pd.DataFrame() # Return empty DataFrame if no CSVs found
        else:
            # Read and concatenate all CSV files
            df_list = []
            for csv_file in all_csv_files:
                print(f"Reading {csv_file}...")
                df_list.append(pd.read_csv(csv_file))

            combined_df = pd.concat(df_list, ignore_index=True)
            print("All CSV files concatenated successfully.")
            return combined_df

In [4]:
# URL for Q4 2016 data
zip_url_q4 = "https://f001.backblazeb2.com/file/Backblaze-Hard-Drive-Data/data_Q4_2016.zip"

# Process the Q1 data using the new function
print("\n--- Processing Q1 2017 Data ---")
df_q4_test = process_backblaze_data(zip_url_q4)


--- Processing Q1 2017 Data ---
Extracting files to temporary directory: /tmp/tmp8t_e73cw
Extraction complete. Identifying and concatenating CSV files...
Reading /tmp/tmp8t_e73cw/2016-10-31.csv...
Reading /tmp/tmp8t_e73cw/2016-12-26.csv...
Reading /tmp/tmp8t_e73cw/2016-10-06.csv...
Reading /tmp/tmp8t_e73cw/2016-11-29.csv...
Reading /tmp/tmp8t_e73cw/2016-11-25.csv...
Reading /tmp/tmp8t_e73cw/2016-10-17.csv...
Reading /tmp/tmp8t_e73cw/2016-10-09.csv...
Reading /tmp/tmp8t_e73cw/2016-11-28.csv...
Reading /tmp/tmp8t_e73cw/2016-12-06.csv...
Reading /tmp/tmp8t_e73cw/2016-11-16.csv...
Reading /tmp/tmp8t_e73cw/2016-12-30.csv...
Reading /tmp/tmp8t_e73cw/2016-10-13.csv...
Reading /tmp/tmp8t_e73cw/2016-11-20.csv...
Reading /tmp/tmp8t_e73cw/2016-12-19.csv...
Reading /tmp/tmp8t_e73cw/2016-10-18.csv...
Reading /tmp/tmp8t_e73cw/2016-12-10.csv...
Reading /tmp/tmp8t_e73cw/2016-11-12.csv...
Reading /tmp/tmp8t_e73cw/2016-10-21.csv...
Reading /tmp/tmp8t_e73cw/2016-10-08.csv...
Reading /tmp/tmp8t_e73cw/201

In [5]:
print("Shape of the full combined DataFrame:")
print(df_q4_test.shape)

Shape of the full combined DataFrame:
(6589448, 95)


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
output_dir = '/content/drive/MyDrive/Colab Notebooks/modele stocastique'
output_file_path = os.path.join(output_dir, 'df_q4_test.csv')

os.makedirs(output_dir, exist_ok=True)
df_q4_test.to_csv(output_file_path, index=False)

print(f"DataFrame successfully saved to: {output_file_path}")

DataFrame successfully saved to: /content/drive/MyDrive/Colab Notebooks/modele stocastique/df_q4_test.csv
